In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"
CHUNKED_LAWS_INDEX_PATH = INDEX_PATH / "chunked_laws_index.pkl"
CHUNKED_COURTS_INDEX_PATH = INDEX_PATH / "chunked_courts_index.pkl"


# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.43 GB)

Index cache: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


# 2. Load Corpora and Build/Load Indices

In [3]:
from bm25index import BM25Index
import bm25index

In [4]:
# Load or build laws index
# Laws CSV: ~45MB, ~269K rows
# Build time: ~30 seconds | Load from cache: <1 second

laws_index = bm25index.get_or_build_index(
    name="laws",
    csv_path=LAWS_CSV,
    index_path=CHUNKED_LAWS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Uncomment to test with smaller corpus
)
print(f"\nLaws index: {len(laws_index.documents):,} documents")

# Test search
test_results = laws_index.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(test_results)

Loading cached laws index from /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/chunked_laws_index.pkl
  Loaded 176,191 documents

Laws index: 176,191 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Vertrag': 1 results
[{'query': 'vertrag', 'hits': [{'rank': 1, 'score': 2.7965, 'text': '4 wird ein rahmen vertrag mit nur einer anbieterin abgeschlossen, so wer den die auf die sem rahmen vertrag beruhenden einzel verträge entsprechend den bedingungen des rahmen vertrags abgeschlossen. für den abschluss der einzel verträge kann die auftraggeberin die jeweilige vertrags partnerin schriftlich auffordern, ihr ange bot zu vervollständigen.', 'citation': 'Art. 25 Abs. 4 BöB'}, {'rank': 2, 'score': 2.7786, 'text': '6 ist das bakom registerbetreiberin, so unter steht der vertrag dem öffentlichen recht (verwaltungsrechtlicher vertrag); ist die auf gabe an einen dritten übertragen, so unter steht der vertrag dem privat recht (privatrechtlicher vertrag).', 'citation': 'Art. 17 Abs. 6 VID'}, {'rank': 3, 'score': 2.7263, 'text': '1 durch vertrag kann die verpflichtung zum abschluss eines künftigen vertrages begründet werden.', 'citation': 'Art. 22 Abs. 1 OR'}]}]


In [5]:
# Load or build courts index
# Courts CSV: ~2.3GB, ~2.5M rows
# Full corpus build time: ~15-20 minutes | Load from cache: ~10 seconds
# Full corpus can have peak memory during build: ~8-16GB

courts_index = bm25index.get_or_build_index(
    name="courts",
    csv_path=COURTS_CSV,
    index_path=CHUNKED_COURTS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Change to use bigger corpus
)
print(f"\nCourts index: {len(courts_index.documents):,} documents")

# Test search
test_results = courts_index.search("Meinungsfreiheit", top_k=3)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")

Loading cached courts index from /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/chunked_courts_index.pkl
  Loaded 2,776,718 documents

Courts index: 2,776,718 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Meinungsfreiheit': 1 results
  Top result: N/A


In [6]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))


In [7]:
import citation_utils
TRAIN_HITS_LAW_PKL = "../data/processed/chunked_search_result_law_hits.pkl"
TRAIN_HITS_COURTS_PKL = "../data/processed/chunked_search_result_courts_hits.pkl"
def get_or_build_train_hits(law_topk=100, court_topk=1000):
    law_hits_l = []
    courts_hits_l = []
    if os.path.exists(TRAIN_HITS_LAW_PKL) and os.path.exists(TRAIN_HITS_COURTS_PKL):
        with open(TRAIN_HITS_LAW_PKL, "rb") as inf:
            law_hits_l = pickle.load(inf)

        with open(TRAIN_HITS_COURTS_PKL, "rb") as inf:
            courts_hits_l = pickle.load(inf)
    else:

        train_df = pd.read_csv("../data/train.csv")
    
        for id, query, gold_citations in tqdm(zip(train_df['query_id'].tolist(), 
                                              train_df['query'].tolist(), 
                                              train_df['gold_citations'].tolist()), 
                                          total=len(train_df), 
                                          desc="train-data") :

            _l = []
            
            test_results = laws_index.search(query, top_k=law_topk)
            _l.extend(test_results[0]['hits'])
        
            test_results = courts_index.search(query, top_k=court_topk)
            _l.extend(test_results[0]['hits'])
    
            first_layer = [hit['citation'] for hit in _l]
            bfs_hits = citation_utils.BFS_citation(court_consideration_d, law_d, first_layer)

            print("bfs_hits.len:", len(bfs_hits), 'first_layer.len:', len(first_layer))

            law_hits = [hit for hit in bfs_hits if hit['citation'] in law_d]
            courts_hits = [hit for hit in bfs_hits if hit['citation'] in court_consideration_d]

            law_hits_l.append(law_hits)
            courts_hits_l.append(courts_hits)
        
        with open(TRAIN_HITS_LAW_PKL, "wb+") as of:
            pickle.dump(law_hits_l, of)

        with open(TRAIN_HITS_COURTS_PKL, "wb+") as of:
            pickle.dump(courts_hits_l, of)

    print(len(law_hits_l), len(law_hits_l[0]))
    return law_hits_l, law_hits_l

In [8]:
def cal_recall(law_hists_l, courts_hits_l, gold_citations_l):
    recalls = []
    for law_hits, courts_hits,gold_citations in zip(law_hists_l, courts_hits_l, gold_citations_l):
        
        all_citation = []
        for hit in law_hits:
            all_citation.append(hit['citation'])
        for hit in courts_hits:
            all_citation.append(hit['citation'])

        hits = len(set(all_citation) & set(gold_citations))

        print("gold_citations.len:", len(gold_citations), ", hits.len:", hits)
        recall = hits / len(gold_citations)
        recalls.append(recall)
        
    mean_recall = np.mean(recalls)
    return mean_recall

In [9]:
law_hits_l, courts_hits_l = get_or_build_train_hits(100, 2000)

train-data:   0%|          | 0/1139 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

train-data: 100%|██████████| 1139/1139 [04:32<00:00,  4.18it/s]

bfs_hits.len: 2673 first_layer.len: 2100


1139 269


In [10]:
train_df = pd.read_csv("../data/train.csv")

print(cal_recall(law_hits_l, courts_hits_l, train_df['gold_citations'].apply(lambda x: x.split(";"))))


gold_citations.len: 3 , hits.len: 0
gold_citations.len: 11 , hits.len: 3
gold_citations.len: 1 , hits.len: 0
gold_citations.len: 4 , hits.len: 4
gold_citations.len: 6 , hits.len: 3
gold_citations.len: 3 , hits.len: 2
gold_citations.len: 3 , hits.len: 2
gold_citations.len: 1 , hits.len: 0
gold_citations.len: 3 , hits.len: 0
gold_citations.len: 14 , hits.len: 2
gold_citations.len: 2 , hits.len: 0
gold_citations.len: 3 , hits.len: 0
gold_citations.len: 5 , hits.len: 1
gold_citations.len: 2 , hits.len: 0
gold_citations.len: 7 , hits.len: 3
gold_citations.len: 9 , hits.len: 7
gold_citations.len: 4 , hits.len: 1
gold_citations.len: 5 , hits.len: 2
gold_citations.len: 4 , hits.len: 0
gold_citations.len: 5 , hits.len: 3
gold_citations.len: 1 , hits.len: 1
gold_citations.len: 1 , hits.len: 0
gold_citations.len: 5 , hits.len: 4
gold_citations.len: 1 , hits.len: 0
gold_citations.len: 2 , hits.len: 1
gold_citations.len: 19 , hits.len: 2
gold_citations.len: 3 , hits.len: 1
gold_citations.len: 1 , h